# 📊 Prévision des Charges d'Impression — Yazaki 2026
### Charges d'Impression par Département · Horizon : Juin – Décembre 2026

| Période | Dates |
|---|---|
| Entraînement | 2023-01 → 2025-12 |
| Test | 2026-01 → 2026-05 |
| Prévision | 2026-06 → 2026-12 |

**Modèles candidats** : LES · Holt · Holt-Winters · SARIMA · Prophet  
**Sélection automatique** par RMSE minimum sur la période de test


## 1. Imports

In [1]:
import sys, os, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

from statsmodels.tsa.holtwinters   import ExponentialSmoothing
from statsmodels.tsa.stattools     import adfuller
from statsmodels.tsa.seasonal      import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf
from pmdarima                      import auto_arima
from sklearn.metrics               import mean_absolute_error, mean_squared_error

print("OK - Librairies importees")

OK - Librairies importees


## 2. Chargement des données

In [2]:
import os
import sys
from pathlib import Path
import pandas as pd

project_root = Path(os.getenv('YAZAKI_PROJECT_ROOT', Path.cwd()))
if not (project_root / 'etl').exists():
    parent = Path.cwd().parent
    if (parent / 'etl').exists():
        project_root = parent

sys.path.insert(0, str(project_root))
from etl.config import get_clean_engine

with get_clean_engine().connect() as conn:
    charges_imp = pd.read_sql("SELECT * FROM ChargesImpression", conn)

print(f"OK - Project root : {project_root}")
print(f"OK - ChargesImpression chargees depuis Yazaki_Clean : {len(charges_imp):,} lignes")
print(charges_imp.head(3))

OK - Project root : c:\Users\habib\Desktop\Yazaki\ETL-Yazaki
OK - ChargesImpression chargees depuis Yazaki_Clean : 9,945 lignes
   ImpressionID NomDepartement CodeDepartement DateImpression TypeImpression  \
0             1            EHS             EHS     2023-01-01     A4-COULEUR   
1             2   PRODUCTION B          PROD-B     2023-01-01          A4-NB   
2             3   PRODUCTION B          PROD-B     2023-01-01     A4-COULEUR   

   NbPages  CoutUnitaire FormatPapier CouleurImpression  DateValid  
0       10         0.156           A4           COULEUR       True  
1       83         0.026           A4     NOIR ET BLANC       True  
2      157         0.156           A4           COULEUR       True  


## 3. Préparation des séries mensuelles par département

In [3]:
charges_imp['DateImpression'] = pd.to_datetime(charges_imp['DateImpression'])
charges_imp['NomDepartement'] = charges_imp['NomDepartement'].astype(str).str.strip()
charges_imp = charges_imp[charges_imp['NomDepartement'].str.upper() != 'INCONNU']
charges_imp['CoutImp'] = charges_imp['NbPages'] * charges_imp['CoutUnitaire']

imp_par_dept = (
    charges_imp
    .assign(Mois=charges_imp['DateImpression'].dt.to_period('M'))
    .groupby(['Mois', 'NomDepartement'])['CoutImp']
    .sum().unstack(fill_value=0).astype(float)
)
imp_par_dept.index = imp_par_dept.index.to_timestamp()

print(f"OK - {imp_par_dept.shape[1]} departements | {len(imp_par_dept)} mois")
print(f"     {imp_par_dept.index[0].date()} -> {imp_par_dept.index[-1].date()}")
print(f"\nDepartements : {sorted(imp_par_dept.columns.tolist())}")

OK - 16 departements | 41 mois
     2023-01-01 -> 2026-05-01

Departements : ['ACHAT', 'COSEE', 'DIRECTION', 'EHS', 'ENGENIERIE', 'FINANCE', 'IT', 'LOGISTIQUE', 'NYS', 'OLS', 'PLPP', 'PRODUCTION A', 'PRODUCTION B', 'QUALITE', 'RH', 'TD']


## 4. Visualisation exploratoire

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Charges d'Impression — Apercu mensuel par departement", fontsize=13, fontweight='bold')

# Top 5 departements
top5 = imp_par_dept.sum().nlargest(5).index
for dept in top5:
    axes[0].plot(imp_par_dept.index, imp_par_dept[dept], marker='o', markersize=3, label=dept)
axes[0].set_title("Top 5 departements")
axes[0].legend(fontsize=8); axes[0].set_ylim(bottom=0)
axes[0].grid(True, alpha=0.4); axes[0].tick_params(axis='x', rotation=20, labelsize=8)

# Total mensuel
total = imp_par_dept.sum(axis=1)
axes[1].fill_between(total.index, total.values, alpha=0.25, color='darkorange')
axes[1].plot(total.index, total.values, color='darkorange', linewidth=2)
axes[1].set_title("Total mensuel — tous departements")
axes[1].set_ylim(bottom=0); axes[1].grid(True, alpha=0.4)
axes[1].tick_params(axis='x', rotation=20, labelsize=8)

plt.tight_layout()
plt.savefig('viz_exploratoire_impression.png', dpi=120, bbox_inches='tight')
plt.show()
print("OK - viz_exploratoire_impression.png")

OK - viz_exploratoire_impression.png


## 5. Analyse statistique des séries

Pour justifier le choix des modèles, 3 outils sont utilisés :
- **Test ADF** : stationnarité de la série
- **Décomposition** : tendance / saisonnalité / résidu
- **ACF** : autocorrélation — détecte la saisonnalité lag 12

In [5]:
def analyser_serie(y, dates, titre, couleur='darkorange', out_dir='analyses_imp'):
    os.makedirs(out_dir, exist_ok=True)
    serie = pd.Series(y, index=dates)

    fig = plt.figure(figsize=(16, 10))
    fig.suptitle(f"Analyse - {titre}", fontsize=13, fontweight='bold')
    gs  = fig.add_gridspec(3, 3, hspace=0.5, wspace=0.35)

    # Serie brute + tendance
    ax1 = fig.add_subplot(gs[0, :2])
    ax1.plot(dates, y, color=couleur, linewidth=1.8, label='Serie brute')
    tendance = serie.rolling(window=6, center=True).mean()
    ax1.plot(dates, tendance, color='red', linewidth=2, linestyle='--', label='Tendance (moy. mobile 6m)')
    ax1.set_title('Serie brute + Tendance')
    ax1.legend(fontsize=8); ax1.set_ylim(bottom=0)
    ax1.grid(True, alpha=0.3); ax1.tick_params(axis='x', rotation=20, labelsize=8)

    # Test ADF
    ax2 = fig.add_subplot(gs[0, 2])
    ax2.axis('off')
    adf    = adfuller(y, autolag='AIC')
    stat   = "OUI" if adf[1] < 0.05 else "NON"
    conseil = "Prete pour ARIMA" if adf[1] < 0.05 else "Differenciation d(1) necessaire"
    texte  = (f"Test ADF - Stationnarite\n{'_'*26}\n"
              f"Statistique : {adf[0]:.3f}\n"
              f"p-value     : {adf[1]:.4f}\n"
              f"Seuil 5%    : {adf[4]['5%']:.3f}\n"
              f"{'_'*26}\n"
              f"Stationnaire : {stat}\n=> {conseil}")
    ax2.text(0.05, 0.95, texte, transform=ax2.transAxes,
             fontsize=9, verticalalignment='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='#f0f9ff', edgecolor='#3b82f6', alpha=0.8))

    # Decomposition
    try:
        decomp = seasonal_decompose(serie, model='additive', period=12, extrapolate_trend='freq')
        for ax, data, title, color in [
            (fig.add_subplot(gs[1, 0]), decomp.trend,    'Tendance',     'red'),
            (fig.add_subplot(gs[1, 1]), decomp.seasonal, 'Saisonnalite', 'green'),
            (fig.add_subplot(gs[1, 2]), decomp.resid,    'Residu',       'purple'),
        ]:
            ax.plot(dates, data, color=color, linewidth=1.5)
            if title == 'Residu': ax.axhline(0, color='black', linewidth=0.8)
            ax.set_title(title); ax.grid(True, alpha=0.3)
            ax.tick_params(axis='x', rotation=20, labelsize=7)
    except Exception as e:
        print(f"  Decomposition non disponible : {e}")

    # ACF
    ax6 = fig.add_subplot(gs[2, :])
    serie_len = len(serie.dropna())
    if serie_len >= 2:
        max_lags = min(24, serie_len - 1)
        plot_acf(serie, lags=max_lags, ax=ax6, color=couleur, alpha=0.05)
        ax6.set_title(f'ACF - Autocorrelogramme (lags={max_lags})')
        if max_lags >= 12:
            ax6.axvline(12, color='red', linestyle=':', linewidth=1.5, label='Lag 12')
            ax6.legend(fontsize=8)
    ax6.set_xlabel('Lag (mois)'); ax6.grid(True, alpha=0.3)

    path = os.path.join(out_dir, f"analyse_{titre.replace(' ','_').replace('/','_')}.png")
    plt.savefig(path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"  Sauvegarde : {path}")

# Top 3 departements impression
print("Analyse des 3 departements les plus charges :")
for dept in imp_par_dept.sum().nlargest(3).index:
    analyser_serie(imp_par_dept[dept].values, imp_par_dept.index, dept)

print("\nOK - Analyses sauvegardees dans analyses_imp/")

Analyse des 3 departements les plus charges :
  Sauvegarde : analyses_imp\analyse_ACHAT.png
  Sauvegarde : analyses_imp\analyse_ENGENIERIE.png
  Sauvegarde : analyses_imp\analyse_TD.png

OK - Analyses sauvegardees dans analyses_imp/


## 6. Modèles de prévision

| Modèle | Librairie | Quand testé |
|---|---|---|
| **LES** | statsmodels | Série stationnaire ou CV < 15% |
| **Holt** | statsmodels | Tendance sans saisonnalité |
| **Holt-Winters** | statsmodels | Saisonnalité détectée + 24 mois min |
| **SARIMA** | pmdarima | Toujours (s'adapte automatiquement) |
| **Prophet** | prophet | Saisonnalité ou forte variabilité (CV > 10%) |

**Sélection** : meilleur RMSE sur la période test (2026-01 → 2026-05)  
**Pré-qualification** : ADF + ACF + CV → seuls les modèles adaptés sont testés


In [6]:
TRAIN_START    = pd.Timestamp('2023-01-01')
TRAIN_END      = pd.Timestamp('2025-12-31')
TEST_START     = pd.Timestamp('2026-01-01')
TEST_END       = pd.Timestamp('2026-05-31')
FORECAST_START = pd.Timestamp('2026-06-01')
FORECAST_END   = pd.Timestamp('2026-12-31')

N_TEST       = 5
N_FORECAST   = 7
WF_MIN_TRAIN = 24


def smape(y_true, y_pred):
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask  = denom > 0
    return np.mean(np.abs(y_true[mask] - y_pred[mask]) / denom[mask]) * 100

def nom_sarima(m):
    p,d,q   = m.order
    P,D,Q,s = m.seasonal_order
    return f"SARIMA({p},{d},{q})({P},{D},{Q})[{s}]" if any([P,D,Q]) else f"ARIMA({p},{d},{q})"


# ═══════════════════════════════════════════════════════════════
# ETAPE 1 : PRE-QUALIFICATION
# ─ ACF lag12 > 0.10  : saisonnalite detectee (seuil bas car impression peu reguliere)
# ─ HW : saisonnalite OU CV >= 0.15 (variabilite moderee suffit)
# ─ Prophet : saisonnalite OU CV >= 0.10
# ─ LES exclu : plat par definition pour h > 1 mois
# ═══════════════════════════════════════════════════════════════
def choisir_modeles(y_train):
    from statsmodels.tsa.stattools import adfuller, acf as acf_fn

    n     = len(y_train)
    mean_ = np.mean(y_train)
    cv    = np.std(y_train) / mean_ if mean_ > 0 else 0

    try:
        stationnaire = adfuller(y_train, autolag='AIC')[1] < 0.05
    except:
        stationnaire = False

    # Seuil 0.10 au lieu de 0.20 : charges impression ont peu de regularite saisonniere
    try:
        saisonniere = abs(acf_fn(y_train, nlags=13, fft=True)[12]) > 0.10
    except:
        saisonniere = False

    modeles = []

    # Holt : tendance sans saisonnalite
    if not saisonniere and cv >= 0.05:
        modeles.append('holt')

    # Holt-Winters : saisonnalite OU CV >= 0.15 + assez de donnees
    # CV >= 0.15 suffit car HW peut capter des patterns meme sans saisonnalite stricte
    if n >= 24 and (saisonniere or cv >= 0.15):
        modeles.append('hw')

    # SARIMA : toujours
    modeles.append('arima')

    # Prophet : saisonnalite OU variabilite moderee
    if n >= 24 and (saisonniere or cv >= 0.10):
        modeles.append('prophet')

    if not modeles:
        modeles = ['arima']

    return modeles, {
        'stationnaire': stationnaire,
        'saisonniere' : saisonniere,
        'cv'          : round(cv, 3),
    }


# ═══════════════════════════════════════════════════════════════
# UTILITAIRE : ENTRAINER + PREDIRE
# ARIMA : d=1 force pour eviter ARIMA(0,0,0) = constante
# ═══════════════════════════════════════════════════════════════
def entrainer_predire(mtype, y_train, n_steps, dates_train=None):
    try:
        if mtype == 'holt':
            m = ExponentialSmoothing(
                    y_train, trend='add', seasonal=None,
                    initialization_method='estimated').fit(optimized=True)
            return np.clip(m.forecast(n_steps), 0, None)

        elif mtype == 'hw':
            m = ExponentialSmoothing(
                    y_train, trend='add', seasonal='add',
                    seasonal_periods=12,
                    initialization_method='estimated').fit(optimized=True)
            return np.clip(m.forecast(n_steps), 0, None)

        elif mtype == 'arima':
            # d=1 force : evite ARIMA(0,0,0) qui produit une prevision constante
            m = auto_arima(
                    y_train, seasonal=True, m=12, stepwise=True,
                    suppress_warnings=True, error_action='ignore',
                    max_p=3, max_q=3, max_P=2, max_Q=2,
                    d=1, max_d=2, max_D=1,
                    information_criterion='aic', with_intercept=True, trend='c')
            return np.clip(m.predict(n_periods=n_steps), 0, None)

        elif mtype == 'prophet':
            from prophet import Prophet as Proph
            if dates_train is None:
                dates_train = pd.date_range(
                    end=pd.Timestamp.today(),
                    periods=len(y_train), freq='MS')
            df_p = pd.DataFrame({'ds': dates_train, 'y': y_train})
            m = Proph(
                yearly_seasonality=True,
                weekly_seasonality=False,
                daily_seasonality=False,
                seasonality_mode='additive',
                changepoint_prior_scale=0.1)
            m.fit(df_p)
            fc = m.predict(m.make_future_dataframe(periods=n_steps, freq='MS'))
            return np.clip(fc['yhat'].values[-n_steps:], 0, None)

    except:
        return None


# ═══════════════════════════════════════════════════════════════
# ETAPE 2 : WALK-FORWARD CROSS-VALIDATION
# Fenetre expansive | h = N_FORECAST mois
# Penalise naturellement les modeles plats sur 7 mois
# Regle unique de selection : min(RMSE moyen)
# ═══════════════════════════════════════════════════════════════
def walk_forward_cv(y, dates, modeles, n_steps=N_FORECAST, min_train=WF_MIN_TRAIN):
    n      = len(y)
    scores = {m: [] for m in modeles}

    for t in range(min_train, n - n_steps + 1):
        y_tr = y[:t]
        y_te = y[t:t + n_steps]
        d_tr = dates[:t] if dates is not None else None

        for mtype in modeles:
            pred = entrainer_predire(mtype, y_tr, n_steps, d_tr)
            if pred is not None and len(pred) == len(y_te):
                scores[mtype].append(np.sqrt(mean_squared_error(y_te, pred)))

    resultats_wf = {
        m: {
            'rmse_moyen': round(np.mean(v), 3),
            'rmse_std'  : round(np.std(v), 3),
            'n_folds'   : len(v),
        }
        for m, v in scores.items() if v
    }

    if not resultats_wf:
        return None, {}

    meilleur = min(resultats_wf, key=lambda m: resultats_wf[m]['rmse_moyen'])
    return meilleur, resultats_wf


# ═══════════════════════════════════════════════════════════════
# ETAPE 3 : FITTED VALUES pour visualisation
# d=1 force aussi ici pour la coherence
# ═══════════════════════════════════════════════════════════════
def calculer_fitted(mtype, y_full, dates_full, n_forecast):
    try:
        if mtype == 'holt':
            mf = ExponentialSmoothing(
                    y_full, trend='add', seasonal=None,
                    initialization_method='estimated').fit(optimized=True)
            return mf.fittedvalues.values

        elif mtype == 'hw':
            mf = ExponentialSmoothing(
                    y_full, trend='add', seasonal='add', seasonal_periods=12,
                    initialization_method='estimated').fit(optimized=True)
            return mf.fittedvalues.values

        elif mtype == 'arima':
            mf = auto_arima(
                    y_full, seasonal=True, m=12, stepwise=True,
                    suppress_warnings=True, error_action='ignore',
                    max_p=3, max_q=3, max_P=2, max_Q=2,
                    d=1, max_d=2, max_D=1,
                    information_criterion='aic', with_intercept=True, trend='c')
            return y_full - mf.resid()

        elif mtype == 'prophet':
            from prophet import Prophet as Proph
            df_p = pd.DataFrame({'ds': dates_full, 'y': y_full})
            mf = Proph(
                yearly_seasonality=True, weekly_seasonality=False,
                daily_seasonality=False, seasonality_mode='additive',
                changepoint_prior_scale=0.1)
            mf.fit(df_p)
            fc = mf.predict(mf.make_future_dataframe(periods=n_forecast, freq='MS'))
            return fc['yhat'].values[:-n_forecast]

    except:
        return np.full(len(y_full), np.mean(y_full))


# ═══════════════════════════════════════════════════════════════
# PIPELINE PRINCIPAL
# Etape 1 : Pre-qualification  -> quels modeles tester
# Etape 2 : Walk-Forward CV    -> meilleur modele (RMSE moyen)
# Etape 3 : Evaluer test fixe  -> RMSE / MAE / sMAPE
# Etape 4 : Forecast final     -> reentrainer sur tout -> 7 mois
# ═══════════════════════════════════════════════════════════════
def prevoir_departement(nom_dept, y_full, dates_full,
                        n_forecast=N_FORECAST, n_test=N_TEST):

    y_train = y_full[:-n_test]
    y_test  = y_full[-n_test:]
    d_train = dates_full[:-n_test]

    modeles, infos = choisir_modeles(y_train)
    print(f"  [{nom_dept}]  stat={infos['stationnaire']}  "
          f"sais={infos['saisonniere']}  CV={infos['cv']:.2f}  "
          f"candidats={modeles}")

    meilleur, scores_wf = walk_forward_cv(
        y_train, d_train, modeles, n_steps=n_forecast)

    if meilleur is None:
        print(f"    WF-CV echoue -> fallback arima")
        meilleur  = 'arima'
        scores_wf = {}

    n_folds = max(0, len(y_train) - WF_MIN_TRAIN - n_forecast + 1)
    print(f"    Walk-Forward CV  (h={n_forecast}, ~{n_folds} folds) :")
    for m, s in sorted(scores_wf.items(), key=lambda x: x[1]['rmse_moyen']):
        tag = '  <-- CHOISI' if m == meilleur else ''
        print(f"      {m:<35} RMSE_moy={s['rmse_moyen']:7.2f}  "
              f"std={s['rmse_std']:5.2f}{tag}")

    pred_test = entrainer_predire(meilleur, y_train, n_test, d_train)
    if pred_test is None:
        pred_test = np.full(n_test, np.mean(y_train[-6:]))

    rmse_test  = np.sqrt(mean_squared_error(y_test, pred_test))
    mae_test   = mean_absolute_error(y_test, pred_test)
    smape_test = smape(y_test, pred_test)
    print(f"    Test fixe        RMSE={rmse_test:.2f}  "
          f"MAE={mae_test:.2f}  sMAPE={smape_test:.1f}%")

    forecast = entrainer_predire(meilleur, y_full, n_forecast, dates_full)
    if forecast is None:
        forecast = np.full(n_forecast, np.mean(y_full[-6:]))

    fitted = calculer_fitted(meilleur, y_full, dates_full, n_forecast)

    future_dates = pd.date_range(
        start=dates_full[-1] + pd.DateOffset(months=1),
        periods=n_forecast, freq='MS')

    comparatif = pd.DataFrame([{
        'Modele'     : m,
        'RMSE_moy'   : s['rmse_moyen'],
        'Std'        : s['rmse_std'],
        'Folds'      : s['n_folds'],
        'Selectionne': 'OUI' if m == meilleur else '',
    } for m, s in scores_wf.items()]).sort_values('RMSE_moy').reset_index(drop=True)

    print(f"  OK [{nom_dept:<16}]  {meilleur:<35}  "
          f"RMSE={rmse_test:.2f}  sMAPE={smape_test:.1f}%")

    return {
        'dept'        : nom_dept,
        'modele'      : meilleur,
        'rmse'        : rmse_test,
        'mae'         : mae_test,
        'smape'       : smape_test,
        'comparatif'  : comparatif,
        'scores_wf'   : scores_wf,
        'forecast'    : forecast,
        'future_dates': future_dates,
        'fitted'      : fitted,
        'pred_test'   : pred_test,
        'y_test'      : y_test,
        'dates'       : dates_full,
        'y'           : y_full,
    }

print("OK - Pipeline charge")
print("  Modeles           : Holt / Holt-Winters / SARIMA / Prophet")
print("  ACF seuil         : 0.10 (adapte aux charges impression)")
print("  HW candidat si    : saisonniere OU CV >= 0.15")
print("  ARIMA             : d=1 force (evite ARIMA(0,0,0) = constante)")
print("  Walk-Forward CV   : h=7 -> min(RMSE moyen)")
print("  Forecast          : 2026-06 -> 2026-12 (7 mois)")


OK - Pipeline charge
  Modeles           : Holt / Holt-Winters / SARIMA / Prophet
  ACF seuil         : 0.10 (adapte aux charges impression)
  HW candidat si    : saisonniere OU CV >= 0.15
  ARIMA             : d=1 force (evite ARIMA(0,0,0) = constante)
  Walk-Forward CV   : h=7 -> min(RMSE moyen)
  Forecast          : 2026-06 -> 2026-12 (7 mois)


## 7. Lancer les prévisions

In [7]:
# Diagnostic variabilite avant lancement
print("=" * 65)
print("  DIAGNOSTIC VARIABILITE — CHARGES IMPRESSION")
print("=" * 65)
for dept in imp_par_dept.columns:
    serie = imp_par_dept[dept].values.astype(float)
    mean_ = np.mean(serie)
    std_  = np.std(serie)
    cv    = std_ / mean_ * 100 if mean_ > 0 else 0
    flag  = "⚠️  TROP PLATE" if cv < 10 else "✅"
    print(f"  {dept:<16} mean={mean_:7.1f}  std={std_:6.1f}  CV={cv:5.1f}%  {flag}")
print("\n=> CV < 10% : relancer le script SQL de variabilite d'abord")

  DIAGNOSTIC VARIABILITE — CHARGES IMPRESSION
  ACHAT            mean=  399.8  std= 119.2  CV= 29.8%  ✅
  COSEE            mean=  108.1  std=  35.1  CV= 32.5%  ✅
  DIRECTION        mean=   56.0  std=  14.6  CV= 26.0%  ✅
  EHS              mean=   43.9  std=  22.5  CV= 51.3%  ✅
  ENGENIERIE       mean=  255.0  std=  50.3  CV= 19.7%  ✅
  FINANCE          mean=   66.6  std=  75.7  CV=113.6%  ✅
  IT               mean=   41.5  std=  37.4  CV= 90.2%  ✅
  LOGISTIQUE       mean=  120.3  std=  20.9  CV= 17.4%  ✅
  NYS              mean=   80.7  std=  41.1  CV= 51.0%  ✅
  OLS              mean=  129.0  std=  27.6  CV= 21.4%  ✅
  PLPP             mean=  162.8  std=  55.7  CV= 34.2%  ✅
  PRODUCTION A     mean=   70.5  std=  17.6  CV= 25.0%  ✅
  PRODUCTION B     mean=  158.0  std=  28.7  CV= 18.1%  ✅
  QUALITE          mean=  188.9  std=  64.1  CV= 33.9%  ✅
  RH               mean=  137.9  std=  42.5  CV= 30.8%  ✅
  TD               mean=  196.3  std=  63.8  CV= 32.5%  ✅

=> CV < 10% : relancer le

In [8]:
print("=" * 65)
print("  CHARGES IMPRESSION — Prevision par departement")
print("=" * 65)

resultats_imp = {}
for dept in imp_par_dept.columns:
    serie = imp_par_dept[dept].astype(float).loc[
        (imp_par_dept.index >= TRAIN_START) & (imp_par_dept.index <= TEST_END)
    ]
    if len(serie) <= N_TEST:
        print(f"  [{dept}] serie trop courte — ignoree")
        continue
    r = prevoir_departement(dept, serie.values, serie.index,
                            n_forecast=N_FORECAST, n_test=N_TEST)
    if r:
        resultats_imp[dept] = r

print(f"\n  => {len(resultats_imp)} departement(s) traite(s)")

  CHARGES IMPRESSION — Prevision par departement
  [ACHAT]  stat=True  sais=True  CV=0.29  candidats=['hw', 'arima', 'prophet']


12:52:42 - cmdstanpy - INFO - Chain [1] start processing
12:52:42 - cmdstanpy - INFO - Chain [1] done processing
12:52:48 - cmdstanpy - INFO - Chain [1] start processing
12:52:48 - cmdstanpy - INFO - Chain [1] done processing
12:52:49 - cmdstanpy - INFO - Chain [1] start processing
12:52:49 - cmdstanpy - INFO - Chain [1] done processing
12:52:51 - cmdstanpy - INFO - Chain [1] start processing
12:52:51 - cmdstanpy - INFO - Chain [1] done processing
12:52:52 - cmdstanpy - INFO - Chain [1] start processing
12:52:52 - cmdstanpy - INFO - Chain [1] done processing
12:52:53 - cmdstanpy - INFO - Chain [1] start processing
12:52:54 - cmdstanpy - INFO - Chain [1] done processing


    Walk-Forward CV  (h=7, ~6 folds) :
      hw                                  RMSE_moy= 135.27  std=11.42  <-- CHOISI
      arima                               RMSE_moy= 157.79  std=38.46
      prophet                             RMSE_moy= 179.14  std=67.11
    Test fixe        RMSE=142.55  MAE=95.31  sMAPE=29.8%
  OK [ACHAT           ]  hw                                   RMSE=142.55  sMAPE=29.8%
  [COSEE]  stat=False  sais=True  CV=0.30  candidats=['hw', 'arima', 'prophet']


12:52:55 - cmdstanpy - INFO - Chain [1] start processing
12:53:08 - cmdstanpy - INFO - Chain [1] done processing
12:53:08 - cmdstanpy - INFO - Chain [1] start processing
12:53:23 - cmdstanpy - INFO - Chain [1] done processing
12:53:23 - cmdstanpy - INFO - Chain [1] start processing
12:53:23 - cmdstanpy - INFO - Chain [1] done processing
12:53:25 - cmdstanpy - INFO - Chain [1] start processing
12:53:25 - cmdstanpy - INFO - Chain [1] done processing
12:53:27 - cmdstanpy - INFO - Chain [1] start processing
12:53:27 - cmdstanpy - INFO - Chain [1] done processing
12:53:28 - cmdstanpy - INFO - Chain [1] start processing
12:53:28 - cmdstanpy - INFO - Chain [1] done processing


    Walk-Forward CV  (h=7, ~6 folds) :
      hw                                  RMSE_moy=  45.27  std= 3.73  <-- CHOISI
      prophet                             RMSE_moy=  48.39  std= 5.75
      arima                               RMSE_moy=  61.14  std= 8.49
    Test fixe        RMSE=43.76  MAE=33.71  sMAPE=45.3%


...

  OK [QUALITE         ]  arima                                RMSE=71.35  sMAPE=46.5%
  [RH]  stat=True  sais=True  CV=0.31  candidats=['hw', 'arima', 'prophet']


12:58:57 - cmdstanpy - INFO - Chain [1] done processing
12:59:01 - cmdstanpy - INFO - Chain [1] start processing
12:59:01 - cmdstanpy - INFO - Chain [1] done processing
12:59:05 - cmdstanpy - INFO - Chain [1] start processing
12:59:06 - cmdstanpy - INFO - Chain [1] done processing
12:59:13 - cmdstanpy - INFO - Chain [1] start processing
12:59:13 - cmdstanpy - INFO - Chain [1] done processing
12:59:17 - cmdstanpy - INFO - Chain [1] start processing
12:59:17 - cmdstanpy - INFO - Chain [1] done processing
12:59:22 - cmdstanpy - INFO - Chain [1] start processing
12:59:22 - cmdstanpy - INFO - Chain [1] done processing


    Walk-Forward CV  (h=7, ~6 folds) :
      hw                                  RMSE_moy=  39.19  std= 1.67  <-- CHOISI
      arima                               RMSE_moy=  43.68  std= 4.33
      prophet                             RMSE_moy=  58.06  std=12.98
    Test fixe        RMSE=51.17  MAE=35.95  sMAPE=27.8%
  OK [RH              ]  hw                                   RMSE=51.17  sMAPE=27.8%
  [TD]  stat=True  sais=False  CV=0.31  candidats=['holt', 'hw', 'arima', 'prophet']


12:59:24 - cmdstanpy - INFO - Chain [1] start processing
12:59:37 - cmdstanpy - INFO - Chain [1] done processing
12:59:38 - cmdstanpy - INFO - Chain [1] start processing
12:59:51 - cmdstanpy - INFO - Chain [1] done processing
12:59:53 - cmdstanpy - INFO - Chain [1] start processing
13:00:07 - cmdstanpy - INFO - Chain [1] done processing
13:00:09 - cmdstanpy - INFO - Chain [1] start processing
13:00:09 - cmdstanpy - INFO - Chain [1] done processing
13:00:11 - cmdstanpy - INFO - Chain [1] start processing
13:00:11 - cmdstanpy - INFO - Chain [1] done processing
13:00:13 - cmdstanpy - INFO - Chain [1] start processing
13:00:13 - cmdstanpy - INFO - Chain [1] done processing


    Walk-Forward CV  (h=7, ~6 folds) :
      holt                                RMSE_moy=  52.03  std= 3.42  <-- CHOISI
      hw                                  RMSE_moy=  55.01  std= 1.91
      arima                               RMSE_moy=  57.81  std=18.67
      prophet                             RMSE_moy=  83.26  std=27.87
    Test fixe        RMSE=69.62  MAE=53.05  sMAPE=37.4%
  OK [TD              ]  holt                                 RMSE=69.62  sMAPE=37.4%

  => 16 departement(s) traite(s)


## 8. Visualisation — Historique · Test · Prévision

Chaque graphique montre **3 zones** :
- **Historique** : données réelles 2023–2025
- **Test** : réel vs prédit sur Jan–Mai 2026
- **Prévision** : Jun–Déc 2026 avec intervalle ±1σ

In [9]:
def visualiser_resultats(resultats, label, c_hist, c_test, c_prev, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    n = len(resultats); cols = 3
    rows = (n + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5))
    axes = axes.flatten() if n > 1 else [axes]
    fig.suptitle(f"Previsions — {label}", fontsize=14, fontweight='bold')

    for ax, (dept, r) in zip(axes, resultats.items()):
        dates   = r['dates']
        y       = r['y']
        n_test  = len(r['y_test'])
        y_train = y[:-n_test]

        # Historique
        ax.plot(dates[:-n_test], y_train, color=c_hist, linewidth=1.8,
                label='Historique', zorder=3)

        # Zone test
        ax.plot(dates[-n_test:], r['y_test'], color=c_hist, linewidth=1.5,
                linestyle='--', zorder=3)
        ax.plot(dates[-n_test:], r['pred_test'], color=c_test, linewidth=2,
                marker='s', markersize=5, linestyle='--', label='Predit (test)', zorder=4)
        ax.fill_between(dates[-n_test:], r['y_test'], r['pred_test'],
                        alpha=0.15, color=c_test)

        # Prevision future + bande confiance
        sigma = np.std(y - r['fitted'])
        ax.plot(r['future_dates'], r['forecast'], color=c_prev, linewidth=2.5,
                marker='o', markersize=5, label='Prevision 2026', zorder=5)
        ax.fill_between(r['future_dates'],
                        np.clip(r['forecast'] - sigma, 0, None),
                        r['forecast'] + sigma,
                        alpha=0.25, color=c_prev, label='+/- 1 sigma')

        # Separateurs
        ax.axvline(dates[-n_test], color='gray',  linestyle=':', linewidth=1.2)
        ax.axvline(dates[-1],      color='black', linestyle=':', linewidth=1.2)

        ax.set_title(f"{dept}\n{r['modele']}\nRMSE={r['rmse']:.1f}  sMAPE={r['smape']:.1f}%",
                     fontsize=8.5, fontweight='bold')
        ax.set_ylim(bottom=0); ax.grid(True, alpha=0.3)
        ax.tick_params(axis='x', labelsize=7, rotation=25)
        ax.legend(fontsize=7, loc='upper left', ncol=2)

    for ax in axes[len(resultats):]:
        ax.set_visible(False)

    plt.tight_layout()
    path = os.path.join(out_dir, 'previsions.png')
    plt.savefig(path, dpi=130, bbox_inches='tight')
    plt.show()
    print(f"OK - Sauvegarde : {path}")

visualiser_resultats(resultats_imp, "Charges Impression",
                     c_hist='darkorange', c_test='royalblue', c_prev='#7c3aed',
                     out_dir='previsions_imp')

OK - Sauvegarde : previsions_imp\previsions.png


## 9. Tableau des performances

In [10]:
def tableau_performances(resultats, label):
    print(f"\n{'='*65}")
    print(f"  {label} — Resume global (trie par RMSE)")
    print(f"{'='*65}")
    rows = [{'Departement': dept, 'Meilleur Modele': r['modele'],
             'RMSE': round(r['rmse'], 2), 'MAE': round(r['mae'], 2),
             'sMAPE%': round(r['smape'], 1)}
            for dept, r in resultats.items()]
    df = pd.DataFrame(rows).set_index('Departement').sort_values('RMSE')
    print(df.to_string())
    print(f"\n  sMAPE moyen : {df['sMAPE%'].mean():.1f}%")
    print(f"\n  Comparatif modeles par departement :")
    for dept, r in resultats.items():
        print(f"\n  [{dept}]")
        print(r['comparatif'].to_string(index=False))
    return df

recap_imp = tableau_performances(resultats_imp, "Charges Impression")


  Charges Impression — Resume global (trie par RMSE)
             Meilleur Modele    RMSE     MAE  sMAPE%
Departement                                         
DIRECTION              arima    7.78    6.88    11.4
LOGISTIQUE              holt   14.90   13.74    10.4
EHS                     holt   19.40   14.54    26.6
PRODUCTION A           arima   23.58   22.88    38.6
OLS                    arima   28.34   25.18    23.5
PRODUCTION B            holt   32.24   29.83    19.1
ENGENIERIE              holt   35.80   22.85     8.1
IT                      holt   41.56   37.50    97.6
COSEE                     hw   43.76   33.71    45.3
NYS                     holt   48.36   38.74    50.3
RH                        hw   51.17   35.95    27.8
TD                      holt   69.62   53.05    37.4
QUALITE                arima   71.35   43.56    46.5
PLPP                    holt   73.98   70.31    48.3
FINANCE                arima  136.85  114.53   117.4
ACHAT                     hw  142.55   95.31 

## 9b. Visualisations avancées — Réel vs Prévisions

In [11]:
def comparaison_reel_prevision(resultats, label, c_reel, c_prev, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    top5 = sorted(resultats.items(), key=lambda x: x[1]['y'].sum(), reverse=True)[:5]

    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    fig.suptitle(f'{label} — Reel vs Previsions', fontsize=13, fontweight='bold')

    for ax, (dept, r) in zip(axes, top5):
        n_test    = len(r['y_test'])
        n_fc      = len(r['forecast'])
        x_test    = np.arange(n_test)
        x_future  = np.arange(n_test, n_test + n_fc)

        ax.bar(x_test - 0.2, r['y_test'],    width=0.4, label='Reel (test)', color=c_reel, alpha=0.8)
        ax.bar(x_test + 0.2, r['pred_test'], width=0.4, label='Predit (test)', color=c_prev, alpha=0.8)
        ax.bar(x_future, r['forecast'], width=0.6, color=c_prev, alpha=0.3,
               edgecolor=c_prev, linewidth=1.5, label=f'Forecast ({n_fc}m)')

        ax.axvline(n_test - 0.5, color='black', linestyle='--', linewidth=1.5, alpha=0.7)
        ax.set_title(f'{dept}\n{r["modele"]}\nsMAPE={r["smape"]:.1f}%', fontsize=8.5, fontweight='bold')
        ax.set_ylabel('Charge (TND)', fontsize=9)

        labels = [f'T{i+1}' for i in range(n_test)] + [f'P{i+1}' for i in range(n_fc)]
        ax.set_xticks(np.arange(n_test + n_fc))
        ax.set_xticklabels(labels, fontsize=7, rotation=45)
        ax.grid(True, alpha=0.2, axis='y')
        ax.legend(fontsize=7)

    plt.tight_layout()
    path = os.path.join(out_dir, 'reel_vs_previsions.png')
    plt.savefig(path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"OK - {path}")

comparaison_reel_prevision(resultats_imp, "Charges Impression",
                           c_reel='darkorange', c_prev='#7c3aed',
                           out_dir='previsions_imp')

OK - previsions_imp\reel_vs_previsions.png


## 10. Enregistrement SQL Server + Export CSV

In [12]:
from sqlalchemy import text
from etl.config import get_dw_engine

def sauvegarder_previsions_sql(resultats, charge_type, date_min=TRAIN_START, date_max=FORECAST_END):
    engine = get_dw_engine()
    rows   = []
    test_dates = pd.date_range(start=TEST_START, end=TEST_END, freq='MS')

    for dept, r in resultats.items():
        for d, v in zip(r['dates'], r['y']):
            rows.append({'NomDepartement': dept, 'Mois': pd.to_datetime(d).date(),
                         'Type': 'Historique', 'ChargeType': charge_type,
                         'ChargeValue': round(float(v), 2), 'Modele': None})

        for d, v in zip(test_dates, r['pred_test']):
            rows.append({'NomDepartement': dept, 'Mois': pd.to_datetime(d).date(),
                         'Type': 'Prevision', 'ChargeType': charge_type,
                         'ChargeValue': round(float(v), 2), 'Modele': str(r.get('modele', ''))})

        for d, v in zip(r['future_dates'], r['forecast']):
            rows.append({'NomDepartement': dept, 'Mois': pd.to_datetime(d).date(),
                         'Type': 'Prevision', 'ChargeType': charge_type,
                         'ChargeValue': round(float(v), 2), 'Modele': str(r.get('modele', ''))})

    if not rows:
        print(f"Aucune ligne a inserer pour {charge_type}")
        return 0

    df_rows = pd.DataFrame(rows)
    with engine.begin() as conn:
        dim_dept = pd.read_sql("SELECT DepartementID, NomDepartement FROM Dim_Departement", conn)
        dept_map = dict(zip(dim_dept['NomDepartement'], dim_dept['DepartementID']))
        df_rows['DepartementID'] = df_rows['NomDepartement'].map(dept_map)

        manquants = sorted(df_rows[df_rows['DepartementID'].isna()]['NomDepartement'].unique().tolist())
        if manquants:
            print(f"Departements absents dans Dim_Departement (ignores) : {manquants}")

        df_rows = df_rows.dropna(subset=['DepartementID']).copy()
        if df_rows.empty:
            print(f"Aucune ligne valide pour {charge_type}")
            return 0

        conn.execute(text("""
            DELETE FROM Previsions
            WHERE ChargeType = :charge_type
              AND Mois BETWEEN :date_min AND :date_max
        """), {'charge_type': charge_type,
                 'date_min': pd.to_datetime(date_min).date(),
                 'date_max': pd.to_datetime(date_max).date()})

        payload = [{'DepartementID': int(row.DepartementID), 'Mois': row.Mois,
                    'Type': row.Type, 'ChargeType': row.ChargeType,
                    'ChargeValue': float(row.ChargeValue),
                    'Modele': (None if pd.isna(row.Modele) or row.Modele == ''
                               else str(row.Modele))}
                   for row in df_rows.itertuples(index=False)]

        conn.execute(text("""
            INSERT INTO Previsions (DepartementID, Mois, Type, ChargeType, ChargeValue, Modele)
            VALUES (:DepartementID, :Mois, :Type, :ChargeType, :ChargeValue, :Modele)
        """), payload)

    print(f"OK - {len(payload)} ligne(s) inseree(s) dans Previsions pour {charge_type}")
    return len(payload)

print("=" * 65)
print("  ENREGISTREMENT SQL SERVER — TABLE Previsions")
print("=" * 65)
n_imp = sauvegarder_previsions_sql(resultats_imp, 'Impression')
print(f"Total insere : {n_imp} ligne(s)")

  ENREGISTREMENT SQL SERVER — TABLE Previsions
OK - 848 ligne(s) inseree(s) dans Previsions pour Impression
Total insere : 848 ligne(s)


In [13]:
from pathlib import Path

def exporter(resultats, fichier):
    rows = []
    test_dates = pd.date_range(start=TEST_START, end=TEST_END, freq='MS')
    for dept, r in resultats.items():
        for d, v in zip(r['dates'], r['y']):
            rows.append({'Departement': dept, 'Mois': d.date(), 'Type': 'Historique',
                         'Charge_TND': round(float(v), 2), 'Modele': ''})
        for d, v in zip(test_dates, r['pred_test']):
            rows.append({'Departement': dept, 'Mois': d.date(), 'Type': 'Prevision_Test',
                         'Charge_TND': round(float(v), 2), 'Modele': r['modele']})
        for d, v in zip(r['future_dates'], r['forecast']):
            rows.append({'Departement': dept, 'Mois': d.date(), 'Type': 'Prevision',
                         'Charge_TND': round(float(v), 2), 'Modele': r['modele']})

    df_export = pd.DataFrame(rows).sort_values(['Departement', 'Mois', 'Type'])
    try:
        df_export.to_csv(fichier, index=False)
        print(f"OK - {fichier}")
        return fichier
    except PermissionError:
        p = Path(fichier)
        fallback = p.with_name(f"{p.stem}_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}{p.suffix}")
        df_export.to_csv(fallback, index=False)
        print(f"WARNING - Fichier verrouille. Export alternatif : {fallback}")
        return str(fallback)

f_imp = exporter(resultats_imp, 'previsions_impression.csv')
print(f"\nFichiers generes :")
print(f"  {f_imp}")
print(f"  previsions_imp/previsions.png")
print(f"  previsions_imp/reel_vs_previsions.png")
print(f"\nTypes disponibles pour Power BI : Historique / Prevision_Test / Prevision")

OK - previsions_impression.csv

Fichiers generes :
  previsions_impression.csv
  previsions_imp/previsions.png
  previsions_imp/reel_vs_previsions.png

Types disponibles pour Power BI : Historique / Prevision_Test / Prevision
